In [1]:
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Base folder containing matchup folders
base_folder = Path(r"C:\Users\James\Documents\MLB\Data\B02. Simulations\1. Game Sims")

# Recursively find all CSV files
all_csvs = list(base_folder.rglob("*.csv"))

print(f"Found {len(all_csvs)} CSV files.")

# Function to read a CSV
def read_csv(file_path):
    try:
        return pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()  # Return empty DF on failure

# Read all CSVs in parallel
with ThreadPoolExecutor() as executor:
    dfs = list(executor.map(read_csv, all_csvs))

# Concatenate all dataframes
combined_df = pd.concat(dfs, ignore_index=True)

print(f"Combined dataframe shape: {combined_df.shape}")


Found 2395 CSV files.
Combined dataframe shape: (7339008, 4)


In [2]:
import pandas as pd
import numpy as np

# Ensure combined_df is sorted
combined_df = combined_df.sort_values(["game_id", "sim"]).reset_index(drop=True)

def analyze_game(df):
    half = len(df) // 2
    
    first_half = df.iloc[:half]
    second_half = df.iloc[half:]
    
    # Determine favorite in first half
    away_wins_first = (first_half['away_score'] > first_half['home_score']).sum()
    home_wins_first = (first_half['home_score'] > first_half['away_score']).sum()
    
    if away_wins_first > home_wins_first:
        favorite = 'away'
        win_rate_first = away_wins_first / half
        win_rate_second = (second_half['away_score'] > second_half['home_score']).sum() / len(second_half)
    elif home_wins_first > away_wins_first:
        favorite = 'home'
        win_rate_first = home_wins_first / half
        win_rate_second = (second_half['home_score'] > second_half['away_score']).sum() / len(second_half)
    else:  # tie
        favorite = 'tie'
        win_rate_first = np.nan
        win_rate_second = np.nan
    
    change = win_rate_second - win_rate_first if not np.isnan(win_rate_first) else np.nan
    increased = change > 0 if not np.isnan(change) else np.nan
    
    return pd.Series({
        "favorite": favorite,
        "win_rate_first_half": win_rate_first,
        "win_rate_second_half": win_rate_second,
        "win_rate_change": change,
        "increased": increased
    })

# Apply per game
result = combined_df.groupby("game_id").apply(analyze_game).reset_index()

print(result.head())


    game_id favorite  win_rate_first_half  win_rate_second_half  \
0  744795.0     home             0.506510              0.506510   
1  744796.0     away             0.580729              0.598307   
2  744797.0     away             0.543620              0.552734   
3  744798.0     away             0.602865              0.595703   
4  744799.0     away             0.574219              0.591797   

   win_rate_change increased  
0         0.000000     False  
1         0.017578      True  
2         0.009115      True  
3        -0.007161     False  
4         0.017578      True  


C:\Users\James\AppData\Local\Temp\ipykernel_16700\4085559519.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = combined_df.groupby("game_id").apply(analyze_game).reset_index()


In [3]:
result['increased'].value_counts(normalize=True)

increased
False    0.53678
True     0.46322
Name: proportion, dtype: float64

In [ ]:
# 1024/512: 0.436387
# 2048/1024: 0.446504
# 3072/1536: 0.46322